# Relational Databases in Python — Review & Practice
### Based on: *Introduction to Importing Data in Python* — "Introduction to relational databases" chapter

This notebook is organized in the same order the chapter teaches the material, so you can use it for **review** (read the summaries) and **testing** (try the questions before checking the Answer Key at the bottom).

**Topic order:**
1. Relational database concepts
2. Creating a database engine in Python (SQLAlchemy)
3. Querying with SQLAlchemy (`connect` → `execute` → `fetch` → `close`)
4. Querying directly with pandas (`read_sql_query`)
5. Advanced querying — JOINing tables

Run the **Setup** cell first — it builds a small practice SQLite database (`Northwind_practice.sqlite`) so every exercise below is actually runnable.

## ⚙️ Setup — build the practice database
Run this once. It creates a mini Northwind-style database with `Orders`, `Customers`, and `Employees` tables, matching the examples used in the chapter.

In [1]:
import sqlite3, os

db_path = "Northwind_practice.sqlite"
if os.path.exists(db_path):
    os.remove(db_path)  # start fresh each run

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("""
CREATE TABLE Orders (
    OrderID INTEGER PRIMARY KEY,
    CustomerID TEXT,
    EmployeeID INTEGER,
    OrderDate TEXT,
    RequiredDate TEXT,
    ShippedDate TEXT,
    ShipVia INTEGER,
    Freight REAL,
    ShipName TEXT,
    ShipAddress TEXT
)
""")

cur.execute("""
CREATE TABLE Customers (
    CustomerID TEXT PRIMARY KEY,
    CompanyName TEXT,
    ContactName TEXT,
    City TEXT,
    Country TEXT
)
""")

cur.execute("""
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    LastName TEXT,
    FirstName TEXT,
    Title TEXT,
    City TEXT
)
""")

orders = [
    (10248, 'VINET', 5, '1996-07-04', '1996-08-01', '1996-07-16', 3, 32.38, 'Vins et alcools Chevalier', "59 rue de l'Abbaye"),
    (10251, 'VICTE', 3, '1996-07-08', '1996-08-05', '1996-07-15', 1, 41.34, 'Victuailles en stock', '2, rue du Commerce'),
    (10254, 'CHOPS', 5, '1996-07-11', '1996-08-08', '1996-07-23', 2, 22.98, 'Chop-suey Chinese', 'Hauptstr. 31'),
    (10256, 'WELLI', 3, '1996-07-15', '1996-08-12', '1996-07-17', 2, 13.97, 'Wellington Importadora', 'Rua do Mercado, 12'),
    (10258, 'ERNSH', 1, '1996-07-17', '1996-08-14', '1996-07-23', 1, 140.51, 'Ernst Handel', 'Kirchgasse 6'),
]
cur.executemany("INSERT INTO Orders VALUES (?,?,?,?,?,?,?,?,?,?)", orders)

customers = [
    ('VINET', 'Vins et alcools Chevalier', 'Paul Henriot', 'Reims', 'France'),
    ('VICTE', 'Victuailles en stock', 'Mary Saveley', 'Lyon', 'France'),
    ('CHOPS', 'Chop-suey Chinese', 'Yang Wang', 'Bern', 'Switzerland'),
    ('WELLI', 'Wellington Importadora', 'Paula Parente', 'Resende', 'Brazil'),
    ('ERNSH', 'Ernst Handel', 'Roland Mendel', 'Graz', 'Austria'),
]
cur.executemany("INSERT INTO Customers VALUES (?,?,?,?,?)", customers)

employees = [
    (1, 'Davolio', 'Nancy', 'Sales Representative', 'Seattle'),
    (2, 'Fuller', 'Andrew', 'Vice President, Sales', 'Tacoma'),
    (3, 'Leverling', 'Janet', 'Sales Representative', 'Kirkland'),
    (5, 'Buchanan', 'Steven', 'Sales Manager', 'London'),
]
cur.executemany("INSERT INTO Employees VALUES (?,?,?,?,?)", employees)

conn.commit()
conn.close()
print("✅ Northwind_practice.sqlite created with Orders, Customers, and Employees tables.")

✅ Northwind_practice.sqlite created with Orders, Customers, and Employees tables.


## Part 1 — Relational Database Concepts
**Summary**
- A *relational database* is based on the **relational model of data**, first described by **Edgar "Ted" Codd**.
- Data lives in multiple **tables** (e.g., `Orders`, `Customers`, `Employees`) that are **linked** to each other through shared key columns (e.g., `Orders.CustomerID` ↔ `Customers.CustomerID`).
- **Codd's 12 Rules** (often called *Codd's 12 Commandments*) actually contain **13 rules**, because they're **zero-indexed** (Rule 0 through Rule 12). They define what a system must do to be considered a true Relational Database Management System (RDBMS).
- Common **RDBMSs**: **PostgreSQL**, **MySQL**, **SQLite**.
- **SQL** = **S**tructured **Q**uery **L**anguage — the language used to query relational databases.

### Q1 (Multiple Choice)
Who first described the relational model of data?

a) Larry Ellison  b) Edgar "Ted" Codd  c) Guido van Rossum  d) Wes McKinney

In [2]:
# Your answer:
answer_q1 = "B"

### Q2 (Multiple Choice)
Codd's 12 Rules are often a trick question. Why?

a) There are actually 10 rules  
b) There are 13 rules, because they're zero-indexed (Rule 0–Rule 12)  
c) There are 12 rules but 3 are optional  
d) There are 24 rules, 12 for tables and 12 for keys

In [3]:
# Your answer:
answer_q2 = "B"

### Q3 (Short Answer)
Name the three Relational Database Management Systems mentioned in the chapter.

In [ ]:
# Your answer:
answer_q3 = ""

### Q4 (True/False)
In the Northwind example, the `Orders` table links to the `Customers` table through the `EmployeeID` column.

In [ ]:
# Your answer (True/False):
answer_q4 = ""

### Q5 (Short Answer)
What does **SQL** stand for?

In [ ]:
# Your answer:
answer_q5 = ""

## Part 2 — Creating a Database Engine in Python
**Summary**
- `SQLAlchemy` works with many different RDBMSs through one consistent interface.
- `create_engine()` creates a connection **engine** to a database.
- For a SQLite file, the connection string format is: `'sqlite:///filename.sqlite'`
- To list the tables in a database: in the original course, `engine.table_names()` was used. **That method is deprecated/removed in modern SQLAlchemy (1.4+/2.0)** — today, use:
```python
from sqlalchemy import inspect
inspect(engine).get_table_names()
```

### Q6 (Code)
Import `create_engine` and create an engine connected to `Northwind_practice.sqlite`.

In [ ]:
# TODO: import create_engine from sqlalchemy
# TODO: create the engine

In [5]:
from sqlalchemy import create_engine
db_path = "Northwind_practice.sqlite"
engine = create_engine(f'sqlite:///{db_path}')

### Q7 (Code)
Print a list of all table names in the database (use the modern `inspect()` approach).

In [8]:
# TODO: list and print all table names
from sqlalchemy import inspect
table_names = inspect(engine).get_table_names()
print(table_names)

['Customers', 'Employees', 'Orders']


### Q8 (Conceptual)
Why might `SQLAlchemy` be preferred over a database-specific library (like `sqlite3` or `psycopg2`) when working with relational databases?

In [ ]:
# Your answer:
answer_q8 = ""

## Part 3 — Querying with SQLAlchemy (the manual workflow)
**Summary — Workflow of SQL querying**
1. Import packages and functions
2. Create the database engine
3. Connect to the engine
4. Query the database
5. Save the query results to a DataFrame
6. Close the connection (or use a `with engine.connect() as con:` **context manager**, which closes it for you automatically)

**Key gotcha:** `pd.DataFrame(rs.fetchall())` does **not** carry over column names — you must set them explicitly:
```python
df.columns = rs.keys()
```
`rs.fetchmany(size=n)` lets you pull only the first `n` rows instead of everything.

**Version note:** In modern SQLAlchemy (1.4+/2.0), `connection.execute()` requires raw SQL strings to be wrapped in `text()`:
```python
from sqlalchemy import text
con.execute(text("SELECT * FROM Orders"))
```

### Q9 (Code)
Using `engine.connect()`, `con.execute()`, and `rs.fetchall()`, query **all rows and columns** from `Orders` into a DataFrame called `df`. Set the column names correctly, then close the connection.

In [ ]:
# TODO: connect, execute "SELECT * FROM Orders", fetchall, set columns, close connection
# Note: modern SQLAlchemy (1.4+/2.0) requires wrapping raw SQL strings in text()
# from sqlalchemy import text


### Q10 (Code)
Now, using the `with` **context manager**, query only the `OrderID`, `OrderDate`, and `ShipName` columns from `Orders`, fetching only the **first 3 rows** with `fetchmany()`.

In [ ]:
# TODO: use 'with engine.connect() as con:' and fetchmany(size=3)
# Remember: wrap the SQL string in text(...)


### Q11 (Debugging / Conceptual)
After running `df = pd.DataFrame(rs.fetchall())`, `df.head()` shows columns labeled `0, 1, 2, 3...` instead of real names. What line fixes this, and why is it needed?

In [ ]:
# Your answer:
answer_q11 = ""

## Part 4 — Querying Directly with pandas
**Summary**
`pandas` offers a shortcut that skips the manual connect → execute → fetchall → close workflow:
```python
df = pd.read_sql_query("SELECT * FROM Orders", engine)
```
This runs the query **and** returns a fully-formed DataFrame (with correct column names already set) in a single line.

### Q12 (Code)
Rewrite your Part 3 query (`SELECT * FROM Orders`) using `pd.read_sql_query()` in one line.

In [ ]:
# TODO: one-line read_sql_query version


### Q13 (Conceptual)
What are the benefits of `pd.read_sql_query()` compared to the manual SQLAlchemy workflow from Part 3?

In [ ]:
# Your answer:
answer_q13 = ""

## Part 5 — Advanced Querying: JOINing Tables
**Summary**
- Related data is split across separate tables, linked by shared key columns (e.g., `Orders.CustomerID` ↔ `Customers.CustomerID`).
- An **INNER JOIN** combines rows from two tables **only where the join condition matches in both tables**.
- Pattern:
```sql
SELECT col1, col2
FROM TableA
INNER JOIN TableB ON TableA.key = TableB.key
```

### Q14 (Code)
Using `pd.read_sql_query()`, write an `INNER JOIN` that returns `OrderID` and `CompanyName` for every order, by joining `Orders` to `Customers` on `CustomerID`.

*Expected first row: `10248 — Vins et alcools Chevalier`*

In [ ]:
# TODO: INNER JOIN Orders + Customers on CustomerID


### Q15 (Code — extension)
Extend the join to **three tables**: return `OrderID`, `CompanyName` (from `Customers`), and `LastName` (from `Employees`) for every order, joining `Orders → Customers` on `CustomerID` **and** `Orders → Employees` on `EmployeeID`.

In [ ]:
# TODO: two-table join chained onto a third table


### Q16 (Conceptual)
If an order's `CustomerID` didn't exist in the `Customers` table, how would that row be treated by an `INNER JOIN` vs. a `LEFT JOIN`?

In [ ]:
# Your answer:
answer_q16 = ""

---
# ✅ Answer Key
**Don't peek until you've attempted every question above!**

**Q1:** b) Edgar "Ted" Codd

**Q2:** b) There are 13 rules, because they're zero-indexed (Rule 0–Rule 12)

**Q3:** PostgreSQL, MySQL, SQLite

**Q4:** False — `Orders` links to `Customers` via `CustomerID`. (It links to `Employees` via `EmployeeID`.)

**Q5:** Structured Query Language

**Q8:** SQLAlchemy provides one consistent interface that works across many different RDBMSs (PostgreSQL, MySQL, SQLite, etc.), so you don't need to learn a separate library/syntax for each database system.

**Q11:** `df.columns = rs.keys()` — `pd.DataFrame(rs.fetchall())` only receives the raw row tuples, not field names, so the column names must be assigned manually from the `ResultProxy`'s `.keys()`.

**Q13:** It's more concise (one line instead of four+), it sets correct column names automatically, and it manages the connection for you.

**Q16:** An `INNER JOIN` would **drop** that order entirely (no match = no row in the result). A `LEFT JOIN` would **keep** the order row, filling the unmatched `Customers` columns with `NULL`/`None`.

### Code Answers

In [ ]:
# Q6 + Q7
from sqlalchemy import create_engine, inspect

engine = create_engine('sqlite:///Northwind_practice.sqlite')

table_names = inspect(engine).get_table_names()
print(table_names)

In [ ]:
# Q9
import pandas as pd

con = engine.connect()
rs = con.execute(text("SELECT * FROM Orders"))
df = pd.DataFrame(rs.fetchall())
df.columns = rs.keys()
con.close()
print(df.head())

In [ ]:
# Q10
with engine.connect() as con:
    rs = con.execute(text("SELECT OrderID, OrderDate, ShipName FROM Orders"))
    df = pd.DataFrame(rs.fetchmany(size=3))
    df.columns = rs.keys()
print(df)

In [ ]:
# Q12
df = pd.read_sql_query("SELECT * FROM Orders", engine)
print(df.head())

In [ ]:
# Q14
df = pd.read_sql_query(
    "SELECT OrderID, CompanyName FROM Orders "
    "INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID",
    engine
)
print(df)

In [ ]:
# Q15
df = pd.read_sql_query(
    "SELECT Orders.OrderID, CompanyName, LastName FROM Orders "
    "INNER JOIN Customers ON Orders.CustomerID = Customers.CustomerID "
    "INNER JOIN Employees ON Orders.EmployeeID = Employees.EmployeeID",
    engine
)
print(df)